# Advanced Problems with Solutions: Chaining and Teeing Iterators

This notebook expands the topic of **chaining and teeing iterators** into a practice-heavy, solution-first workbook.

Main tools:

- `itertools.chain`
- `itertools.chain.from_iterable`
- `itertools.tee`
- `yield from`
- safe iterator design patterns

The problems are intentionally more advanced than basic syntax exercises. They focus on **lazy evaluation**, **single-pass streams**, **iterator exhaustion**, and **memory-aware use of `tee`**.


## Best-practice checklist

Use this checklist while solving the problems:

1. Prefer `chain.from_iterable(iterable_of_iterables)` when the outer iterable should stay lazy.
2. Avoid `chain(*iterables)` when `iterables` could be huge, expensive, or infinite, because unpacking is eager.
3. Treat iterators as **single-use** objects.
4. Use `tee` only when each derived iterator is consumed at roughly the same pace.
5. Avoid `tee` on a very large or infinite iterator when one branch may run far ahead of another.
6. Prefer storing data in a list/tuple when the input is small and you truly need repeated passes.
7. Add tests with partial consumption, because iterator bugs often appear only after some values have already been consumed.


In [1]:
from itertools import (
    chain,
    tee,
    islice,
    count,
    takewhile,
    dropwhile,
    zip_longest,
)
from collections import deque
from dataclasses import dataclass
from typing import Iterable, Iterator, Callable, TypeVar, Any

T = TypeVar("T")
U = TypeVar("U")


def take(n: int, iterable: Iterable[T]) -> list[T]:
    """Return the first n items from any iterable."""
    return list(islice(iterable, n))


def consume(iterable: Iterable[Any]) -> None:
    """Exhaust an iterable without keeping its values."""
    deque(iterable, maxlen=0)


print("Setup complete.")


Setup complete.


## Warm-up: what problem does `chain` solve?

A common pattern is receiving several iterables and wanting to process them as a single stream.

The key idea: `chain(a, b, c)` does **not** copy the values from `a`, `b`, and `c`; it pulls from each iterable only when needed.


In [2]:
morning = (f"morning-{i}" for i in range(3))
afternoon = (f"afternoon-{i}" for i in range(2))
evening = (f"evening-{i}" for i in range(4))

stream = chain(morning, afternoon, evening)

take(5, stream)


['morning-0', 'morning-1', 'morning-2', 'afternoon-0', 'afternoon-1']

In [3]:
# The stream continued from where we stopped.
list(stream)


['evening-0', 'evening-1', 'evening-2', 'evening-3']

## Warm-up: why `chain.from_iterable` matters

When you have an **iterable of iterables**, use `chain.from_iterable`.

This preserves laziness in the outer iterable.


In [4]:
def noisy_batches():
    print("creating batch 1")
    yield (1, 2, 3)
    print("creating batch 2")
    yield (4, 5)
    print("creating batch 3")
    yield (6, 7, 8, 9)


lazy_flat = chain.from_iterable(noisy_batches())

print("Nothing printed after construction except this line.")
print("First two values:", take(2, lazy_flat))
print("Next three values:", take(3, lazy_flat))


Nothing printed after construction except this line.
creating batch 1
First two values: [1, 2]
creating batch 2
Next three values: [3, 4, 5]


## Warm-up: why unpacking can destroy laziness

`chain(*some_iterable)` must unpack the entire outer iterable before `chain` can be called.

That means `chain(*outer)` is **not safe** for very large or infinite outer iterables.


In [5]:
def noisy_outer():
    for i in range(3):
        print(f"yielding inner iterable {i}")
        yield range(i * 10, i * 10 + 3)


print("Using chain(*outer):")
eager_flat = chain(*noisy_outer())  # all inner iterables are requested immediately
print("Now consuming:")
print(list(eager_flat))


Using chain(*outer):
yielding inner iterable 0
yielding inner iterable 1
yielding inner iterable 2
Now consuming:
[0, 1, 2, 10, 11, 12, 20, 21, 22]


In [6]:
print("Using chain.from_iterable(outer):")
lazy_flat = chain.from_iterable(noisy_outer())  # no inner iterable requested yet
print("Now consuming only 4 values:")
print(take(4, lazy_flat))


Using chain.from_iterable(outer):
Now consuming only 4 values:
yielding inner iterable 0
yielding inner iterable 1
[0, 1, 2, 10]


## Warm-up: what problem does `tee` solve?

`tee` creates multiple iterators from a single iterable.

They can be consumed independently, but internally `tee` may buffer values when one branch moves ahead of the other.


In [7]:
source = (i * i for i in range(6))
a, b = tee(source, 2)

print("a first 3:", take(3, a))
print("b first 2:", take(2, b))
print("a rest:", list(a))
print("b rest:", list(b))


a first 3: [0, 1, 4]
b first 2: [0, 1]
a rest: [9, 16, 25]
b rest: [4, 9, 16, 25]


## `tee` is not a free copy

This example traces the original source. Notice that when one tee branch advances, the original source advances. The slower branch later receives buffered values.


In [8]:
def traced_numbers(n):
    for i in range(n):
        print(f"source produced {i}")
        yield i


left, right = tee(traced_numbers(5), 2)

print("Advance left three times:")
print(next(left))
print(next(left))
print(next(left))

print("Advance right once:")
print(next(right))

print("Advance left again:")
print(next(left))

print("Finish right:")
print(list(right))


Advance left three times:
source produced 0
0
source produced 1
1
source produced 2
2
Advance right once:
0
Advance left again:
source produced 3
3
Finish right:
source produced 4
[1, 2, 3, 4]


# Problem 1 — Recreate `chain`

Implement `my_chain(*iterables)`.

Requirements:

- Accept any number of iterables.
- Yield every item from the first iterable, then the second, and so on.
- Do not materialize the inputs.
- Use `yield from`.
- Pass the tests.


In [9]:
# Problem 1 starter

def my_chain(*iterables):
    # TODO: replace with your implementation
    raise NotImplementedError


## Solution 1


In [10]:
def my_chain(*iterables):
    for iterable in iterables:
        yield from iterable


assert list(my_chain([1, 2], (), range(3), "ab")) == [1, 2, 0, 1, 2, "a", "b"]

g1 = (i for i in range(2))
g2 = (i for i in range(2, 4))
joined = my_chain(g1, g2)

assert next(joined) == 0
assert list(joined) == [1, 2, 3]

print("Solution 1 passed.")


Solution 1 passed.


# Problem 2 — Recreate `chain.from_iterable`

Implement `my_chain_from_iterable(iterables)`.

Requirements:

- Accept a single iterable that produces inner iterables.
- Preserve laziness in the outer iterable.
- Use `yield from`.
- Prove laziness with a trace list.


In [11]:
# Problem 2 starter

def my_chain_from_iterable(iterables):
    # TODO: replace with your implementation
    raise NotImplementedError


## Solution 2


In [12]:
def my_chain_from_iterable(iterables):
    for iterable in iterables:
        yield from iterable


events = []

def outer():
    events.append("outer started")
    yield [1, 2]
    events.append("second requested")
    yield [3]
    events.append("third requested")
    yield [4, 5]


flat = my_chain_from_iterable(outer())

assert events == []                 # construction is lazy
assert next(flat) == 1
assert events == ["outer started"]   # only first inner iterable requested
assert list(flat) == [2, 3, 4, 5]
assert events == ["outer started", "second requested", "third requested"]

print("Solution 2 passed.")


Solution 2 passed.


# Problem 3 — Flatten paginated API results lazily

You receive API pages where each page contains a list of records.

Build `iter_records(fetch_pages)`.

Requirements:

- `fetch_pages()` returns an iterator of pages.
- Each page is a list of dictionaries.
- Yield one record at a time.
- Do not build a list of all records.
- Use `chain.from_iterable`.


In [13]:
# Problem 3 starter

def iter_records(fetch_pages):
    # TODO: replace with your implementation
    raise NotImplementedError


## Solution 3


In [14]:
def iter_records(fetch_pages):
    return chain.from_iterable(fetch_pages())


def fake_pages():
    print("fetch page 1")
    yield [{"id": 1}, {"id": 2}]
    print("fetch page 2")
    yield [{"id": 3}]
    print("fetch page 3")
    yield [{"id": 4}, {"id": 5}]


records = iter_records(fake_pages)

print("Only one record requested:")
print(next(records))

print("Next two records:")
print(take(2, records))

print("Remaining records:")
print(list(records))


Only one record requested:
fetch page 1
{'id': 1}
Next two records:
fetch page 2
[{'id': 2}, {'id': 3}]
Remaining records:
fetch page 3
[{'id': 4}, {'id': 5}]


In [15]:
def quiet_pages():
    yield [{"id": 1}, {"id": 2}]
    yield []
    yield [{"id": 3}]

assert [r["id"] for r in iter_records(quiet_pages)] == [1, 2, 3]

print("Solution 3 passed.")


Solution 3 passed.


# Problem 4 — Fix an eager flattening bug

The following function is risky:

```python
def flatten_bad(iterables):
    return chain(*iterables)
```

It is risky because `*iterables` consumes the entire outer iterable immediately.

Write `flatten_good(iterables)` that keeps the outer iterable lazy.


In [16]:
# Problem 4 starter

def flatten_good(iterables):
    # TODO: replace with your implementation
    raise NotImplementedError


## Solution 4


In [17]:
def flatten_good(iterables):
    return chain.from_iterable(iterables)


events = []

def infinite_style_outer():
    for i in count():
        events.append(f"outer requested {i}")
        yield [i]


flat = flatten_good(infinite_style_outer())

assert events == []
assert take(5, flat) == [0, 1, 2, 3, 4]
assert events == [
    "outer requested 0",
    "outer requested 1",
    "outer requested 2",
    "outer requested 3",
    "outer requested 4",
]

print("Solution 4 passed.")


Solution 4 passed.


# Problem 5 — Chain headers, rows, and footers without copying

You need to export report lines.

Build `report_lines(header, rows, footer)`.

Requirements:

- `header` is a string.
- `rows` is any iterable of strings.
- `footer` is a string.
- Return a lazy iterator producing header, rows, footer.
- Do not convert `rows` to a list.


In [18]:
# Problem 5 starter

def report_lines(header, rows, footer):
    # TODO: replace with your implementation
    raise NotImplementedError


## Solution 5


In [19]:
def report_lines(header, rows, footer):
    return chain([header], rows, [footer])


def row_generator():
    for i in range(3):
        print(f"making row {i}")
        yield f"row-{i}"


lines = report_lines("BEGIN", row_generator(), "END")

print(next(lines))
print(next(lines))
print(list(lines))


BEGIN
making row 0
row-0
making row 1
making row 2
['row-1', 'row-2', 'END']


In [20]:
assert list(report_lines("H", (str(i) for i in range(2)), "F")) == ["H", "0", "1", "F"]

print("Solution 5 passed.")


Solution 5 passed.


# Problem 6 — Combine multiple log streams

You have log streams from several services. Each stream yields lines already sorted by service-local time, but you only need to read them sequentially service by service.

Build `combined_logs(*streams)`.

Requirements:

- Return a lazy iterator.
- Skip empty streams naturally.
- Do not use nested lists.
- Use `chain`.


In [21]:
# Problem 6 starter

def combined_logs(*streams):
    # TODO: replace with your implementation
    raise NotImplementedError


## Solution 6


In [22]:
def combined_logs(*streams):
    return chain(*streams)


api_logs = (f"api:{i}" for i in range(2))
worker_logs = iter(())
db_logs = (f"db:{i}" for i in range(3))

assert list(combined_logs(api_logs, worker_logs, db_logs)) == [
    "api:0", "api:1", "db:0", "db:1", "db:2"
]

print("Solution 6 passed.")


Solution 6 passed.


# Problem 7 — Build `nwise` with `tee`

Implement `nwise(iterable, n)`.

For example:

```python
list(nwise([1, 2, 3, 4], 3))
# [(1, 2, 3), (2, 3, 4)]
```

Requirements:

- Use `tee`.
- Use `islice` to offset the cloned iterators.
- Return an iterator of tuples.
- Raise `ValueError` when `n < 1`.


In [23]:
# Problem 7 starter

def nwise(iterable, n):
    # TODO: replace with your implementation
    raise NotImplementedError


## Solution 7


In [24]:
def nwise(iterable, n):
    if n < 1:
        raise ValueError("n must be at least 1")

    iterators = tee(iterable, n)
    shifted = [islice(iterator, offset, None) for offset, iterator in enumerate(iterators)]
    return zip(*shifted)


assert list(nwise([1, 2, 3, 4], 1)) == [(1,), (2,), (3,), (4,)]
assert list(nwise([1, 2, 3, 4], 2)) == [(1, 2), (2, 3), (3, 4)]
assert list(nwise([1, 2, 3, 4], 3)) == [(1, 2, 3), (2, 3, 4)]
assert list(nwise([1, 2], 3)) == []

try:
    list(nwise([1, 2], 0))
    raise AssertionError("Expected ValueError")
except ValueError:
    pass

print("Solution 7 passed.")


Solution 7 passed.


# Problem 8 — Pairwise deltas from a single-pass stream

Implement `pairwise_deltas(values)`.

It should return differences between consecutive values:

```python
list(pairwise_deltas([10, 13, 11, 20]))
# [3, -2, 9]
```

Requirements:

- Use `tee`.
- Consume the input only once.
- Work with generators.


In [25]:
# Problem 8 starter

def pairwise_deltas(values):
    # TODO: replace with your implementation
    raise NotImplementedError


## Solution 8


In [26]:
def pairwise_deltas(values):
    left, right = tee(values, 2)
    next(right, None)
    return (b - a for a, b in zip(left, right))


assert list(pairwise_deltas([10, 13, 11, 20])) == [3, -2, 9]
assert list(pairwise_deltas(iter([5]))) == []
assert list(pairwise_deltas(x * 10 for x in range(5))) == [10, 10, 10, 10]

print("Solution 8 passed.")


Solution 8 passed.


# Problem 9 — Detect local peaks

A value is a local peak when it is greater than both its immediate neighbors.

Implement `local_peaks(values)`.

Example:

```python
list(local_peaks([1, 3, 2, 4, 5, 1, 2]))
# [3, 5]
```

Requirements:

- Use `nwise(values, 3)` from Problem 7.
- Return a lazy iterator.


In [27]:
# Problem 9 starter

def local_peaks(values):
    # TODO: replace with your implementation
    raise NotImplementedError


## Solution 9


In [28]:
def local_peaks(values):
    return (middle for left, middle, right in nwise(values, 3) if middle > left and middle > right)


assert list(local_peaks([1, 3, 2, 4, 5, 1, 2])) == [3, 5]
assert list(local_peaks([1, 2, 3])) == []
assert list(local_peaks([3, 2, 1])) == []

print("Solution 9 passed.")


Solution 9 passed.


# Problem 10 — Split a stream at the first matching item

Implement `split_at_first(iterable, predicate)`.

It should return two iterators:

1. `before`: values before the first item that satisfies `predicate`
2. `from_match`: the first matching value and everything after it

Example:

```python
before, rest = split_at_first([1, 2, 3, 4, 5], lambda x: x >= 3)
list(before)  # [1, 2]
list(rest)    # [3, 4, 5]
```

Requirements:

- Use `tee`.
- Use `takewhile` and `dropwhile`.
- Avoid materializing the whole stream.


In [29]:
# Problem 10 starter

def split_at_first(iterable, predicate):
    # TODO: replace with your implementation
    raise NotImplementedError


## Solution 10


In [30]:
def split_at_first(iterable, predicate):
    a, b = tee(iterable, 2)
    before = takewhile(lambda x: not predicate(x), a)
    from_match = dropwhile(lambda x: not predicate(x), b)
    return before, from_match


before, rest = split_at_first([1, 2, 3, 4, 5], lambda x: x >= 3)
assert list(before) == [1, 2]
assert list(rest) == [3, 4, 5]

before, rest = split_at_first((i for i in range(5)), lambda x: x == 99)
assert list(before) == [0, 1, 2, 3, 4]
assert list(rest) == []

print("Solution 10 passed.")


Solution 10 passed.


## Important note about Problem 10

The solution is lazy, but it can create internal `tee` buffering if one side is consumed far ahead of the other.

For finite streams this is often acceptable. For large or infinite streams, prefer a custom stateful splitter when possible, or consume both sides in a controlled way.


# Problem 11 — Fork a stream into transformed branches

Implement `fork_map(iterable, *functions)`.

It should return one iterator per function. Each returned iterator applies its function to the same original stream.

Example:

```python
numbers = range(4)
squares, cubes = fork_map(numbers, lambda x: x*x, lambda x: x**3)
list(squares)  # [0, 1, 4, 9]
list(cubes)    # [0, 1, 8, 27]
```

Requirements:

- Use `tee`.
- Return a tuple of iterators.
- Be aware of `tee` buffering when one branch runs far ahead.


In [31]:
# Problem 11 starter

def fork_map(iterable, *functions):
    # TODO: replace with your implementation
    raise NotImplementedError


## Solution 11


In [32]:
def fork_map(iterable, *functions):
    copies = tee(iterable, len(functions))
    return tuple(map(func, copy) for func, copy in zip(functions, copies))


squares, cubes, labels = fork_map(
    range(5),
    lambda x: x * x,
    lambda x: x ** 3,
    lambda x: f"value={x}",
)

assert list(squares) == [0, 1, 4, 9, 16]
assert list(cubes) == [0, 1, 8, 27, 64]
assert list(labels) == ["value=0", "value=1", "value=2", "value=3", "value=4"]

print("Solution 11 passed.")


Solution 11 passed.


# Problem 12 — Build a two-pass report from a single-pass stream

You receive a generator of sales transactions. You need:

- total revenue
- all suspicious transactions over a threshold

Implement `sales_report(transactions, suspicious_threshold)`.

Requirements:

- The input may be a generator.
- Use `tee` to create two branches.
- Return a dictionary with keys `"total_revenue"` and `"suspicious"`.
- For this problem, it is acceptable to materialize only the suspicious transactions.


In [33]:
# Problem 12 starter

def sales_report(transactions, suspicious_threshold):
    # TODO: replace with your implementation
    raise NotImplementedError


## Solution 12


In [34]:
def sales_report(transactions, suspicious_threshold):
    for_total, for_suspicious = tee(transactions, 2)

    total_revenue = sum(txn["amount"] for txn in for_total)
    suspicious = [
        txn
        for txn in for_suspicious
        if txn["amount"] >= suspicious_threshold
    ]

    return {
        "total_revenue": total_revenue,
        "suspicious": suspicious,
    }


def transaction_stream():
    yield {"id": "A", "amount": 100}
    yield {"id": "B", "amount": 2_500}
    yield {"id": "C", "amount": 75}
    yield {"id": "D", "amount": 4_000}


report = sales_report(transaction_stream(), suspicious_threshold=1_000)

assert report == {
    "total_revenue": 6675,
    "suspicious": [
        {"id": "B", "amount": 2500},
        {"id": "D", "amount": 4000},
    ],
}

print("Solution 12 passed.")


Solution 12 passed.


## Better design discussion for Problem 12

The `tee` solution is useful for practicing duplicated iterator branches.

However, a one-pass version is usually better here because it avoids `tee` buffering:

```python
total = 0
suspicious = []
for txn in transactions:
    total += txn["amount"]
    if txn["amount"] >= threshold:
        suspicious.append(txn)
```

Best practice: use `tee` when it simplifies a pipeline, but do not force it when a direct single-pass loop is clearer and more memory-stable.


# Problem 13 — Look ahead without losing the original stream

Implement `with_next(iterable, default=None)`.

It should yield `(current, next_value)` pairs.

Example:

```python
list(with_next([10, 20, 30], default="END"))
# [(10, 20), (20, 30), (30, "END")]
```

Requirements:

- Use `tee`.
- Return a lazy iterator.
- Work with generators.


In [35]:
# Problem 13 starter

def with_next(iterable, default=None):
    # TODO: replace with your implementation
    raise NotImplementedError


## Solution 13


In [36]:
def with_next(iterable, default=None):
    current, nxt = tee(iterable, 2)
    next(nxt, None)
    return zip_longest(current, nxt, fillvalue=default)


assert list(with_next([10, 20, 30], default="END")) == [
    (10, 20),
    (20, 30),
    (30, "END"),
]

assert list(with_next([], default="END")) == []

print("Solution 13 passed.")


Solution 13 passed.


# Problem 14 — Parse sections from chained lines

You receive lines from multiple files, but for this exercise each file is represented by a list of strings.

A section starts with a line like:

```text
[users]
```

or

```text
[orders]
```

Build `section_items(files)`.

Requirements:

- Chain the lines from all files.
- Ignore blank lines and comments starting with `#`.
- Yield `(section, value)` for normal lines.
- A normal line belongs to the most recent section.
- Use `chain.from_iterable`.


In [37]:
# Problem 14 starter

def section_items(files):
    # TODO: replace with your implementation
    raise NotImplementedError


## Solution 14


In [38]:
def section_items(files):
    current_section = None

    for raw_line in chain.from_iterable(files):
        line = raw_line.strip()

        if not line or line.startswith("#"):
            continue

        if line.startswith("[") and line.endswith("]"):
            current_section = line[1:-1]
            continue

        if current_section is None:
            raise ValueError(f"Value without section: {line!r}")

        yield current_section, line


files = [
    ["# file 1", "[users]", "alice", "bob"],
    ["", "[orders]", "order-1"],
    ["order-2", "# done"],
]

assert list(section_items(files)) == [
    ("users", "alice"),
    ("users", "bob"),
    ("orders", "order-1"),
    ("orders", "order-2"),
]

print("Solution 14 passed.")


Solution 14 passed.


# Problem 15 — Lazy CSV-like row pipeline

You receive row batches. Each row is a dictionary.

Build `valid_emails(row_batches)`.

Requirements:

- Flatten row batches lazily.
- Ignore rows without an `"email"` key.
- Normalize email to lowercase and strip whitespace.
- Keep only strings containing `"@"`.
- Use `chain.from_iterable`.


In [39]:
# Problem 15 starter

def valid_emails(row_batches):
    # TODO: replace with your implementation
    raise NotImplementedError


## Solution 15


In [40]:
def valid_emails(row_batches):
    rows = chain.from_iterable(row_batches)

    for row in rows:
        raw_email = row.get("email")
        if not isinstance(raw_email, str):
            continue

        email = raw_email.strip().lower()
        if "@" in email:
            yield email


batches = [
    [{"email": " ALICE@EXAMPLE.COM "}, {"name": "No Email"}],
    [{"email": "bad-email"}, {"email": "bob@example.com"}],
    [],
    [{"email": None}, {"email": "carol@Example.org"}],
]

assert list(valid_emails(batches)) == [
    "alice@example.com",
    "bob@example.com",
    "carol@example.org",
]

print("Solution 15 passed.")


Solution 15 passed.


# Problem 16 — A memory-aware warning demo for `tee`

This problem is conceptual and diagnostic.

Build a demonstration showing that when one tee branch runs far ahead, the source is advanced and the slower branch must later receive old values from an internal buffer.

Requirements:

- Use a traced source.
- Create two tee branches.
- Consume the fast branch ahead of the slow branch.
- Explain what happened in comments.


In [41]:
# Problem 16 starter

def tee_buffering_demo():
    # TODO: replace with your demonstration
    raise NotImplementedError


## Solution 16


In [42]:
def tee_buffering_demo():
    events = []

    def source():
        for i in range(6):
            events.append(f"source produced {i}")
            yield i

    slow, fast = tee(source(), 2)

    fast_values = take(5, fast)
    events_after_fast = list(events)

    slow_values = take(2, slow)
    events_after_slow = list(events)

    return {
        "fast_values": fast_values,
        "events_after_fast": events_after_fast,
        "slow_values": slow_values,
        "events_after_slow": events_after_slow,
    }


demo = tee_buffering_demo()

assert demo["fast_values"] == [0, 1, 2, 3, 4]
assert demo["events_after_fast"] == [
    "source produced 0",
    "source produced 1",
    "source produced 2",
    "source produced 3",
    "source produced 4",
]

# The slow branch gets 0 and 1 without producing new source values.
assert demo["slow_values"] == [0, 1]
assert demo["events_after_slow"] == demo["events_after_fast"]

demo


{'fast_values': [0, 1, 2, 3, 4],
 'events_after_fast': ['source produced 0',
  'source produced 1',
  'source produced 2',
  'source produced 3',
  'source produced 4'],
 'slow_values': [0, 1],
 'events_after_slow': ['source produced 0',
  'source produced 1',
  'source produced 2',
  'source produced 3',
  'source produced 4']}

# Problem 17 — Merge static and generated task queues

You have three sources of task IDs:

1. urgent tasks: a finite list
2. scheduled tasks: a generator
3. backfill tasks: a finite tuple

Build `task_queue(urgent, scheduled, backfill)`.

Requirements:

- Urgent tasks must come first.
- Scheduled tasks must be lazy.
- Backfill tasks must come last.
- Use `chain`.


In [43]:
# Problem 17 starter

def task_queue(urgent, scheduled, backfill):
    # TODO: replace with your implementation
    raise NotImplementedError


## Solution 17


In [44]:
def task_queue(urgent, scheduled, backfill):
    return chain(urgent, scheduled, backfill)


def scheduled_tasks():
    for i in range(3):
        print(f"creating scheduled task {i}")
        yield f"scheduled-{i}"


queue = task_queue(["urgent-1", "urgent-2"], scheduled_tasks(), ("backfill-1",))

print(next(queue))
print(next(queue))
print(list(queue))


urgent-1
urgent-2
creating scheduled task 0
creating scheduled task 1
creating scheduled task 2
['scheduled-0', 'scheduled-1', 'scheduled-2', 'backfill-1']


In [45]:
assert list(task_queue(["u"], (f"s{i}" for i in range(2)), ("b",))) == ["u", "s0", "s1", "b"]

print("Solution 17 passed.")


Solution 17 passed.


# Problem 18 — Chain generators produced by a factory

Build `chain_factories(*factories)`.

Each factory is a zero-argument function returning an iterable. The function should call one factory at a time only when needed.

Requirements:

- Do not call all factories up front.
- Preserve laziness.
- Yield all values from the first factory's iterable, then the second factory's iterable, and so on.


In [46]:
# Problem 18 starter

def chain_factories(*factories):
    # TODO: replace with your implementation
    raise NotImplementedError


## Solution 18


In [47]:
def chain_factories(*factories):
    for factory in factories:
        yield from factory()


events = []

def make_a():
    events.append("make_a called")
    return iter([1, 2])

def make_b():
    events.append("make_b called")
    return iter([3, 4])

stream = chain_factories(make_a, make_b)

assert events == []
assert next(stream) == 1
assert events == ["make_a called"]
assert next(stream) == 2
assert next(stream) == 3
assert events == ["make_a called", "make_b called"]
assert list(stream) == [4]

print("Solution 18 passed.")


Solution 18 passed.


# Problem 19 — Group adjacent equal values with `tee`

Implement `change_points(iterable)`.

It should yield `(index, previous_value, new_value)` each time the value changes compared with the previous item.

Example:

```python
list(change_points(["A", "A", "B", "B", "C", "A"]))
# [(2, "A", "B"), (4, "B", "C"), (5, "C", "A")]
```

Requirements:

- Use `tee`.
- Work lazily.
- Do not materialize the whole input.


In [48]:
# Problem 19 starter

def change_points(iterable):
    # TODO: replace with your implementation
    raise NotImplementedError


## Solution 19


In [49]:
def change_points(iterable):
    previous, current = tee(iterable, 2)
    next(current, None)

    for index, (old, new) in enumerate(zip(previous, current), start=1):
        if old != new:
            yield index, old, new


assert list(change_points(["A", "A", "B", "B", "C", "A"])) == [
    (2, "A", "B"),
    (4, "B", "C"),
    (5, "C", "A"),
]

assert list(change_points([])) == []
assert list(change_points(["A"])) == []

print("Solution 19 passed.")


Solution 19 passed.


# Problem 20 — Capstone: streaming event pipeline

You receive events in pages. Each page is a list of dictionaries.

Build `event_pipeline(fetch_pages)` returning a dictionary with:

- `"first_five_ids"`: the first five event IDs
- `"error_ids"`: all IDs where `level == "ERROR"`
- `"level_transitions"`: adjacent level changes as `(index, old_level, new_level)`

Requirements:

- Use `chain.from_iterable` to flatten pages.
- Use `tee` to create branches from the flattened event stream.
- Use your `change_points` helper for level transitions.
- Keep the solution simple and readable.
- It is acceptable to materialize the final small output lists.


In [50]:
# Problem 20 starter

def event_pipeline(fetch_pages):
    # TODO: replace with your implementation
    raise NotImplementedError


## Solution 20


In [51]:
def event_pipeline(fetch_pages):
    events = chain.from_iterable(fetch_pages())

    for_first, for_errors, for_levels = tee(events, 3)

    first_five_ids = [event["id"] for event in islice(for_first, 5)]

    error_ids = [
        event["id"]
        for event in for_errors
        if event.get("level") == "ERROR"
    ]

    levels = (event.get("level") for event in for_levels)
    level_transitions = list(change_points(levels))

    return {
        "first_five_ids": first_five_ids,
        "error_ids": error_ids,
        "level_transitions": level_transitions,
    }


def fetch_event_pages():
    yield [
        {"id": "e1", "level": "INFO"},
        {"id": "e2", "level": "INFO"},
        {"id": "e3", "level": "WARNING"},
    ]
    yield [
        {"id": "e4", "level": "ERROR"},
        {"id": "e5", "level": "ERROR"},
    ]
    yield [
        {"id": "e6", "level": "INFO"},
        {"id": "e7", "level": "ERROR"},
    ]


result = event_pipeline(fetch_event_pages)

assert result == {
    "first_five_ids": ["e1", "e2", "e3", "e4", "e5"],
    "error_ids": ["e4", "e5", "e7"],
    "level_transitions": [
        (2, "INFO", "WARNING"),
        (3, "WARNING", "ERROR"),
        (5, "ERROR", "INFO"),
        (6, "INFO", "ERROR"),
    ],
}

result


{'first_five_ids': ['e1', 'e2', 'e3', 'e4', 'e5'],
 'error_ids': ['e4', 'e5', 'e7'],
 'level_transitions': [(2, 'INFO', 'WARNING'),
  (3, 'WARNING', 'ERROR'),
  (5, 'ERROR', 'INFO'),
  (6, 'INFO', 'ERROR')]}

# Extra drills

These are additional practice prompts. Try them before reading the hints.

1. Write `chain_non_empty(iterables)` that skips empty inner iterables while preserving laziness.
2. Write `prepend_many(prefix_items, iterable)` using `chain`.
3. Write `append_many(iterable, suffix_items)` using `chain`.
4. Write `surround(prefix_items, iterable, suffix_items)` using `chain`.
5. Write `first_duplicate(iterable)` using a single pass and no `tee`.
6. Write `moving_average(values, window_size)` using `nwise`.
7. Write `validate_sorted(iterable)` using `tee` to compare adjacent values.
8. Write `line_numbers(files)` that chains file lines and yields `(line_number, line)`.
9. Write `tee_three_ways(iterable)` and consume the branches in lockstep.
10. Rewrite a `tee`-based solution as a one-pass loop and compare clarity.


## Extra drill solutions


In [52]:
def chain_non_empty(iterables):
    # This naturally skips empty inner iterables because yielding from an empty iterable yields nothing.
    return chain.from_iterable(iterables)


def prepend_many(prefix_items, iterable):
    return chain(prefix_items, iterable)


def append_many(iterable, suffix_items):
    return chain(iterable, suffix_items)


def surround(prefix_items, iterable, suffix_items):
    return chain(prefix_items, iterable, suffix_items)


def first_duplicate(iterable):
    seen = set()
    for item in iterable:
        if item in seen:
            return item
        seen.add(item)
    return None


def moving_average(values, window_size):
    if window_size < 1:
        raise ValueError("window_size must be at least 1")

    for window in nwise(values, window_size):
        yield sum(window) / window_size


def validate_sorted(iterable):
    left, right = tee(iterable, 2)
    next(right, None)
    return all(a <= b for a, b in zip(left, right))


def line_numbers(files):
    lines = chain.from_iterable(files)
    return enumerate(lines, start=1)


def tee_three_ways(iterable):
    return tee(iterable, 3)


assert list(chain_non_empty([[1], [], [2, 3]])) == [1, 2, 3]
assert list(prepend_many(["H"], [1, 2])) == ["H", 1, 2]
assert list(append_many([1, 2], ["F"])) == [1, 2, "F"]
assert list(surround(["H"], [1, 2], ["F"])) == ["H", 1, 2, "F"]
assert first_duplicate([1, 2, 3, 2, 4]) == 2
assert first_duplicate([1, 2, 3]) is None
assert list(moving_average([10, 20, 30, 40], 2)) == [15, 25, 35]
assert validate_sorted([1, 2, 2, 3]) is True
assert validate_sorted([1, 3, 2]) is False
assert list(line_numbers([["a", "b"], ["c"]])) == [(1, "a"), (2, "b"), (3, "c")]

x, y, z = tee_three_ways(range(3))
assert list(x) == [0, 1, 2]
assert list(y) == [0, 1, 2]
assert list(z) == [0, 1, 2]

print("Extra drill solutions passed.")


Extra drill solutions passed.


# Common mistakes and fixes

| Mistake | Why it is a problem | Better approach |
|---|---|---|
| `chain(list_of_iterables)` | Yields each inner iterable, not each inner value | `chain.from_iterable(list_of_iterables)` |
| `chain(*generator_of_iterables)` | Eagerly consumes the outer generator | `chain.from_iterable(generator_of_iterables)` |
| Reusing the same generator twice | The second pass may be empty | Use `tee`, or materialize a small finite input |
| Using `tee` when one branch runs far ahead | Internal buffer can grow large | Consume branches together or use a one-pass loop |
| Using `tee` for simple aggregation | Often more complex than needed | Prefer a direct loop when it is clearer |
| Assuming `tee` copies values deeply | It duplicates iteration, not objects | Copy mutable values explicitly if needed |


# Final best-practice summary

- `chain` is for a known set of iterables.
- `chain.from_iterable` is for an iterable that produces iterables.
- `yield from` is the cleanest way to implement custom chaining behavior.
- `tee` is useful but should be treated as a buffering adapter, not a magical free copy.
- When in doubt, write a small trace generator and test partial consumption.
